# CTMatch Classifier Retraining

Retrains the 3-class relevance classifier for clinical trial matching.

**Problem**: `semaj83/scibert_finetuned_pruned_ctmatch` predicts *relevant* for ~80% of
TREC topics despite only ~22% being truly relevant — a 3.6× bias that drives 70%+ of
pipeline errors.

**Fixes applied here**:
- **Base model**: PubMedBERT (`microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`)
  — stronger biomedical pretraining than SciBERT
- **Loss**: Focal loss (γ=2) with inverse-frequency class weights — suppresses overconfident
  easy-negative predictions and upweights the minority relevant/partial classes
- **Eval**: TREC NDCG@10 inline after training — the downstream metric that matters

**Architecture**: Cross-encoder. Input is `[CLS] patient description [SEP] trial text [SEP]`.
The model attends jointly over both, unlike bi-encoders which compare embeddings independently.

**Label set**: `0=not_relevant`, `1=partially_relevant`, `2=relevant`

## 1. Environment

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime > Change runtime type > GPU before training')

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn scipy tqdm lxml
!pip install -q git+https://github.com/semajyllek/ctmatch.git

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # paste token — never commit this

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

All paths and hyperparameters live in two dataclasses.
Update `DataConfig.data_root` to match your Drive layout.

**Model options** (set `TrainConfig.base_model`):

| Model | Params | Notes |
|-------|--------|-------|
| `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` | 110M | **Default** |
| `dmis-lab/bioelectra-base-discriminator-pubmed` | 110M | ELECTRA — faster convergence |
| `sultan/BioM-ELECTRA-Large-SQuAD2` | 340M | Larger — better but needs A100 |

In [ ]:
from dataclasses import dataclass
import os

@dataclass
class DataConfig:
    data_root: str = '/content/drive/MyDrive/ct_data23'
    hf_dataset: str = 'semaj83/ctmatch_classification'
    hf_data_file: str = 'combined_classifier_data.jsonl'
    max_length: int = 512
    train_frac: float = 0.8
    val_frac: float = 0.1
    seed: int = 42

    @property
    def trec_root(self): return os.path.join(self.data_root, 'evaluation', 'trec_data')

    @property
    def kz_root(self): return os.path.join(self.data_root, 'evaluation', 'kz_data')

    @property
    def model_dir(self): return os.path.join(self.data_root, 'models')

    @property
    def results_file(self): return os.path.join(self.data_root, 'clf_results.json')


@dataclass
class TrainConfig:
    base_model: str = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
    num_labels: int = 3
    learning_rate: float = 2e-5
    batch_size: int = 16
    num_epochs: int = 6
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    focal_gamma: float = 2.0
    fp16: bool = True
    hub_model_id: str = ''  # e.g. 'your_username/ctmatch-clf-v1'

    def save_dir(self, data_cfg: DataConfig) -> str:
        slug = self.base_model.split('/')[-1]
        return os.path.join(data_cfg.model_dir, f'clf_{slug}')

    def run_id(self) -> str:
        from datetime import datetime
        slug = self.base_model.split('/')[-1]
        ts = datetime.now().strftime('%Y%m%d_%H%M')
        return f'{slug}__{ts}'


# ── run flags ────────────────────────────────────────────────────────────
RUN_TRAIN     = True   # set False to load an existing checkpoint
RUN_TREC_EVAL = True   # compute NDCG@10 on TREC + KZ qrels after training
PUSH_TO_HUB   = False  # push trained model to HF Hub

data_cfg  = DataConfig()
train_cfg = TrainConfig()
print(f'Data root:  {data_cfg.data_root}')
print(f'Base model: {train_cfg.base_model}')
print(f'Save dir:   {train_cfg.save_dir(data_cfg)}')
print(f'Results:    {data_cfg.results_file}')

In [ ]:
import json
from dataclasses import asdict

def load_results(cfg: DataConfig) -> dict:
    if os.path.exists(cfg.results_file):
        with open(cfg.results_file) as f:
            return json.load(f)
    return {}

def save_result(run_id: str, metrics: dict, train_cfg: TrainConfig, cfg: DataConfig) -> None:
    results = load_results(cfg)
    results[run_id] = {
        'model':       train_cfg.base_model,
        'lr':          train_cfg.learning_rate,
        'epochs':      train_cfg.num_epochs,
        'batch_size':  train_cfg.batch_size,
        'focal_gamma': train_cfg.focal_gamma,
        **metrics,
    }
    with open(cfg.results_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Saved → {cfg.results_file}  (total runs: {len(results)})')

print(f'Existing runs in {data_cfg.results_file}:')
existing = load_results(data_cfg)
print(f'  {list(existing.keys()) or "none yet"}')

## 3. Data

Training data: `semaj83/ctmatch_classification` — ~40K (topic, doc, label) triples.
Labels are stored as strings `'0'/'1'/'2'`; we convert them to integers during tokenization.

Class distribution is ~4.5:1:1 (not_relevant : partial : relevant).
The focal loss + class weights address this imbalance without resampling.

In [ ]:
from datasets import load_dataset
from ctmatch.utils.ctmatch_utils import train_test_val_split
import pandas as pd
from collections import Counter

LABEL_NAMES = ['not_relevant', 'partially_relevant', 'relevant']

def load_and_split(cfg: DataConfig):
    raw = load_dataset(cfg.hf_dataset, data_files=cfg.hf_data_file)
    splits = {'train': cfg.train_frac, 'val': cfg.val_frac}
    return train_test_val_split(raw, splits, seed=cfg.seed)

def class_distribution(dataset) -> pd.DataFrame:
    rows = []
    for split_name, split in dataset.items():
        labels = [int(l) for l in split['label']]
        counts = Counter(labels)
        total = len(split)
        row = {'split': split_name, 'total': total}
        for i, name in enumerate(LABEL_NAMES):
            row[name] = counts.get(i, 0)
            row[f'{name}_%'] = f"{100 * counts.get(i, 0) / total:.1f}%"
        rows.append(row)
    return pd.DataFrame(rows).set_index('split')

dataset = load_and_split(data_cfg)
dist = class_distribution(dataset)
print(dist[['not_relevant', 'partially_relevant', 'relevant', 'total']])

In [ ]:
def preview_raw_examples(dataset, n_per_class: int = 1):
    """Print one example per label class so the data format is clear."""
    train = dataset['train']
    seen = {}
    for i in range(len(train)):
        label_int = int(train[i]['label'])
        if label_int not in seen:
            seen[label_int] = i
        if len(seen) == 3:
            break
    for label_int in sorted(seen):
        i = seen[label_int]
        topic = train[i]['topic']
        doc   = train[i]['doc']
        print(f"{'─'*72}")
        print(f"Label: {label_int}  ({LABEL_NAMES[label_int]})")
        print(f"Topic: {topic[:300]}")
        print(f"  Doc: {doc[:300]}")
    print(f"{'─'*72}")

preview_raw_examples(dataset)

In [ ]:
import numpy as np
import torch

def compute_class_weights(train_split, num_labels: int = 3) -> torch.Tensor:
    """Complement-frequency weights: w_i = 1 - freq_i, then normalize."""
    labels = [int(l) for l in train_split['label']]
    counts = np.bincount(labels, minlength=num_labels).astype(float)
    weights = 1.0 - counts / counts.sum()
    return torch.tensor(weights / weights.sum(), dtype=torch.float32).to(device)

class_weights = compute_class_weights(dataset['train'])
for name, w in zip(LABEL_NAMES, class_weights.cpu().tolist()):
    print(f'  {name}: {w:.4f}')

## 4. Model

PubMedBERT as a sequence-pair classifier (cross-encoder). The tokenizer produces
`[CLS] topic [SEP] doc [SEP]`, with `truncation='longest_first'` so neither field
is completely truncated when one is unusually long.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ID2LABEL = {0: 'not_relevant', 1: 'partially_relevant', 2: 'relevant'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

def build_model_and_tokenizer(cfg: TrainConfig):
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.base_model,
        num_labels=cfg.num_labels,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    return model.to(device), tokenizer

model, tokenizer = build_model_and_tokenizer(train_cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params / 1e6:.1f}M')

In [ ]:
def make_tokenize_fn(tokenizer, cfg: DataConfig):
    def tokenize(batch):
        out = tokenizer(
            batch['topic'], batch['doc'],
            max_length=cfg.max_length,
            truncation='longest_first',
            padding='max_length',
        )
        # labels stored as strings '0'/'1'/'2' — convert to int
        out['labels'] = [int(l) for l in batch['label']]
        return out
    return tokenize

tokenize_fn = make_tokenize_fn(tokenizer, data_cfg)
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['doc', 'topic', 'label'])
# Note: no set_format('torch') — the Trainer's default collator converts Python lists to
# tensors at batch time, avoiding a torchvision.io.VideoReader import bug in some Colab envs
print(tokenized)

In [ ]:
def preview_tokenized_example(tokenized_ds, tokenizer):
    """Show what one tokenized example looks like — keys, length, and decoded text."""
    # .with_format(None) returns plain Python lists, avoiding the torchvision import
    # that set_format('torch') triggers when indexing on some Colab environments
    ex = tokenized_ds['train'].with_format(None)[0]
    label_int = ex['labels']
    n_real = sum(ex['attention_mask'])
    n_pad  = len(ex['input_ids']) - n_real

    print(f"Keys:    {list(ex.keys())}")
    print(f"Label:   {label_int}  ({LABEL_NAMES[label_int]})")
    print(f"Tokens:  {n_real} real + {n_pad} padding = {len(ex['input_ids'])} total")
    print()

    real_ids = ex['input_ids'][:n_real]
    decoded  = tokenizer.decode(real_ids, skip_special_tokens=False)

    print("Decoded real tokens (first 500 chars):")
    print(decoded[:500])
    if len(decoded) > 500:
        print(f"  ... [{len(decoded) - 500} more chars] ...")
        print("  ... (last 150 chars) ...")
        print(decoded[-150:])

preview_tokenized_example(tokenized, tokenizer)

## 5. Training

**FocalLossTrainer** wraps HuggingFace `Trainer` with two changes:

- **Class weights** in the cross-entropy loss upweight `partially_relevant` and `relevant`
- **Focal modulation** `(1 - p_correct)^γ` down-weights easy correct predictions

Net effect: the model stops spending gradient steps overconfidently confirming obvious
negatives, and instead focuses on the hard boundary between partial and relevant.

In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import torch.nn.functional as F
import math

class FocalLossTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, gamma: float = 2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # **kwargs absorbs extra arguments added in newer transformers (e.g. num_items_in_batch)
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        ce = F.cross_entropy(logits, labels, weight=self.class_weights, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma * ce).mean()
        return (loss, outputs) if return_outputs else loss

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'f1_macro':    round(f1_score(labels, preds, average='macro'), 4),
        'f1_weighted': round(f1_score(labels, preds, average='weighted'), 4),
    }

In [ ]:
def build_trainer(
    model, tokenizer, tokenized_ds,
    class_weights: torch.Tensor,
    train_cfg: TrainConfig,
    data_cfg: DataConfig,
) -> FocalLossTrainer:
    n_train = len(tokenized_ds['train'])
    steps_per_epoch = math.ceil(n_train / train_cfg.batch_size)
    warmup_steps = int(steps_per_epoch * train_cfg.num_epochs * train_cfg.warmup_ratio)

    args = TrainingArguments(
        output_dir=train_cfg.save_dir(data_cfg),
        num_train_epochs=train_cfg.num_epochs,
        learning_rate=train_cfg.learning_rate,
        per_device_train_batch_size=train_cfg.batch_size,
        per_device_eval_batch_size=train_cfg.batch_size,
        weight_decay=train_cfg.weight_decay,
        warmup_steps=warmup_steps,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        fp16=train_cfg.fp16,
        logging_steps=steps_per_epoch,
        report_to='none',
    )

    return FocalLossTrainer(
        class_weights=class_weights,
        gamma=train_cfg.focal_gamma,
        model=model,
        args=args,
        train_dataset=tokenized_ds['train'],
        eval_dataset=tokenized_ds['validation'],
        processing_class=tokenizer,  # renamed from 'tokenizer' in transformers >= 4.47
        compute_metrics=compute_metrics,
    )

trainer = build_trainer(model, tokenizer, tokenized, class_weights, train_cfg, data_cfg)

In [ ]:
if RUN_TRAIN:
    trainer.train()
    save_path = train_cfg.save_dir(data_cfg)
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f'Model saved → {save_path}')
else:
    save_path = train_cfg.save_dir(data_cfg)
    print(f'Loading checkpoint from {save_path}')
    model = AutoModelForSequenceClassification.from_pretrained(save_path).to(device)
    tokenizer = AutoTokenizer.from_pretrained(save_path)
    trainer.model = model  # keep trainer usable for test metrics

## 6. Evaluation

### 6a. Held-out test set

Confusion matrix and per-class F1 on the 20% held-out split of the HF dataset.
Watch for improvement in recall for `relevant` (row 2) — the previous model had 49% recall there.

In [ ]:
def show_test_metrics(trainer, tokenized_ds) -> dict:
    output = trainer.predict(tokenized_ds['test'])
    y_pred = output.predictions.argmax(-1)
    y_true = output.label_ids

    print('=== Confusion matrix (rows=actual, cols=predicted) ===')
    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES)
    print(df_cm)
    print()
    print('=== Classification report ===')
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

    return {
        'test_f1_macro':    round(f1_score(y_true, y_pred, average='macro'), 4),
        'test_f1_weighted': round(f1_score(y_true, y_pred, average='weighted'), 4),
        'test_confusion_matrix': cm.tolist(),
    }

test_metrics = show_test_metrics(trainer, tokenized)

### 6b. TREC NDCG@10

Score all judged (topic, doc) pairs using the classifier alone and rank by P(relevant).
This measures *classifier quality in isolation* — useful for comparing models without
running the full retrieval pipeline.

> **Note**: This number is not directly comparable to the pipeline NDCG in `eval_baseline.ipynb`
> (which also runs sim filter and SVM). After pushing the model to HF Hub, run the full
> pipeline eval by setting `classifier_model_checkpoint` in `eval_baseline.ipynb`.

In [ ]:
from ctmatch.evaluation.eval_utils import (
    get_trec_topic2text, get_kz_topic2text, calc_ndcg, calc_first_positive_rank,
    load_eval_datasets, load_doc_texts,
)

if RUN_TREC_EVAL:
    eval_datasets = load_eval_datasets(data_cfg.trec_root, data_cfg.kz_root)
    doc_texts = load_doc_texts()

In [ ]:
from tqdm.auto import tqdm

def score_judged_docs(
    topic_text: str,
    doc_ids: list,
    doc_texts: dict,
    model,
    tokenizer,
    max_length: int,
    batch_size: int = 32,
) -> dict:
    """Returns {doc_id: P(relevant)} for all judged docs under this topic."""
    model.eval()
    scores = {}
    for i in range(0, len(doc_ids), batch_size):
        batch_ids = doc_ids[i:i+batch_size]
        batch_docs = [doc_texts.get(d, '') for d in batch_ids]
        enc = tokenizer(
            [topic_text] * len(batch_ids), batch_docs,
            max_length=max_length,
            truncation='longest_first',
            padding=True,
            return_tensors='pt',
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 2].cpu().tolist()  # P(relevant)
        scores.update(zip(batch_ids, probs))
    return scores

def eval_on_dataset(ds_dict: dict, model, tokenizer, doc_texts: dict, cfg: DataConfig) -> dict:
    topic2text = ds_dict['topic2text']
    rel_dict = ds_dict['rel_dict']
    ndcgs, mrrs = [], []
    for topic_id, doc2rel in tqdm(rel_dict.items(), desc='topics'):
        if topic_id not in topic2text:
            continue
        judged_ids = list(doc2rel.keys())
        scores = score_judged_docs(
            topic2text[topic_id], judged_ids, doc_texts, model, tokenizer, cfg.max_length
        )
        ranked = sorted(scores, key=scores.get, reverse=True)
        ndcgs.append(calc_ndcg(ranked, doc2rel))
        _, mrr = calc_first_positive_rank(ranked, doc2rel)
        mrrs.append(mrr)
    return {
        'ndcg@10': round(np.mean(ndcgs), 4),
        'mrr':     round(np.mean(mrrs), 4),
        'n_topics': len(ndcgs),
    }

In [ ]:
if RUN_TREC_EVAL:
    trec_results = {}
    for ds_name, ds in eval_datasets.items():
        print(f'\nEvaluating {ds_name}...')
        trec_results[ds_name] = eval_on_dataset(ds, model, tokenizer, doc_texts, data_cfg)

    print('\n=== TREC Classifier-Only NDCG@10 ===')
    df_res = pd.DataFrame(trec_results).T[['ndcg@10', 'mrr', 'n_topics']]
    print(df_res)
    print('\nBaseline (full pipeline sim+svm+clf): NDCG@10=0.6525, MRR=0.3050')

    # flatten trec results for storage: trec21_ndcg@10, trec21_mrr, etc.
    flat_trec = {f'{ds}_{k}': v for ds, res in trec_results.items() for k, v in res.items()}
    all_metrics = {**test_metrics, **flat_trec}
else:
    all_metrics = {**test_metrics}

run_id = train_cfg.run_id()
save_result(run_id, all_metrics, train_cfg, data_cfg)
print(f'\nRun ID: {run_id}')

## 7. Save and Publish

After pushing to HF Hub, update `eval_baseline.ipynb` to use the new model:

```python
ctm = CTMatch(PipeConfig(
    ir_setup=True,
    classifier_model_checkpoint='your_username/ctmatch-clf-v1'
))
```

Note: `dataprep.py` has a `SUPPORTED_LMS` allowlist that needs updating for
new checkpoint paths — patch it or add your model ID to the list.

In [ ]:
save_path = train_cfg.save_dir(data_cfg)
print(f'Model checkpoint: {save_path}')

if PUSH_TO_HUB and train_cfg.hub_model_id:
    model.push_to_hub(train_cfg.hub_model_id, token=os.environ['HF_TOKEN'])
    tokenizer.push_to_hub(train_cfg.hub_model_id, token=os.environ['HF_TOKEN'])
    print(f'Pushed → https://huggingface.co/{train_cfg.hub_model_id}')
else:
    print('Set PUSH_TO_HUB=True and train_cfg.hub_model_id to publish.')

## 8. Results Comparison

Load all saved runs from Drive and display as a sortable table.
Run this cell any time — it reads the JSON without needing to retrain.

In [ ]:
all_results = load_results(data_cfg)

if not all_results:
    print('No results yet — run the eval cells first.')
else:
    cols = [
        'model', 'epochs', 'lr', 'focal_gamma',
        'test_f1_macro', 'test_f1_weighted',
        'trec21_ndcg@10', 'trec22_ndcg@10', 'kz_ndcg@10',
        'trec21_mrr', 'trec22_mrr', 'kz_mrr',
    ]
    df_all = pd.DataFrame(all_results).T.reset_index(names='run_id')
    # only show columns that exist (trec cols absent if RUN_TREC_EVAL was False)
    show_cols = ['run_id'] + [c for c in cols if c in df_all.columns]
    df_show = df_all[show_cols].copy()

    # shorten model names for readability
    df_show['model'] = df_show['model'].str.split('/').str[-1]

    print(df_show.to_string(index=False))